# Advanced Weather Classification: Ensemble & Optimization

This notebook extends the base transfer learning notebook (MH1) with advanced techniques:
1. **Data Augmentation + Early Stopping** — fix ResNet18 overfitting (was 5.32% gap)
2. **Fine-tuning VGG16** — unfreeze last conv block to push past 95%
3. **Ensemble** — average both models to exploit complementary class strengths
4. **Knowledge Distillation** — compress ensemble into a lightweight student CNN
5. **Pruning + Fine-tuning** — VGG16 at 70% sparsity with accuracy recovery

**Run on Google Colab with T4 GPU runtime.** Export this notebook as `.ipynb` from Databricks (File → Export → IPython Notebook).

## Setup & Installation

We install `torchsummary` for model architecture summaries, download the **Multi-class Weather Dataset** (1125 images across 4 classes: cloudy, rain, shine, sunrise), and import all required libraries.

PyTorch and torchvision come pre-installed on Colab. We also detect GPU availability automatically.

In [0]:
!pip install -q torchsummary
!wget -q http://cdn.iiith.talentsprint.com/aiml/Experiment_related_data/Multi-classWeatherDataset.zip
!unzip -qq -o Multi-classWeatherDataset.zip

import torch
import torchvision
from torch import nn, optim
import torch.nn.functional as F
from torchsummary import summary
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np
import time, copy, os, tempfile, itertools
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report
import torch.nn.utils.prune as prune

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Data Loading with Augmentation + Utility Functions

**Key improvement over MH1:** We apply data augmentation **only to training data**:
- `RandomHorizontalFlip` — mirrors images to double effective dataset size
- `RandomRotation(15°)` — teaches model rotation invariance
- `ColorJitter` — simulates different lighting conditions (crucial for weather!)

Validation and test sets keep standard transforms for unbiased evaluation.

We also define reusable utilities used across all sections:
- **`EarlyStopping`** — monitors val loss, saves best model, stops when no improvement for `patience` epochs
- **`train_one_epoch()`** — supports single or multiple optimizers (needed for Section 2)
- **`evaluate_model()`** — computes loss and accuracy on any data split
- **`get_predictions()`** — collects all predictions for confusion matrices
- **`plot_cm()`** — visualizes prediction patterns with counts and percentages

In [0]:
# ==================== Transforms ====================
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ==================== Dataset Split ====================
base_dataset = datasets.ImageFolder(root='Multi-class Weather Dataset')
class_names = base_dataset.classes
print(f"Classes: {class_names}")
print(f"Total images: {len(base_dataset)}")

train_size = int(0.70 * len(base_dataset))
val_size = int(0.15 * len(base_dataset))
test_size = len(base_dataset) - train_size - val_size

# Custom subset to apply different transforms per split
class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, base_ds, indices, transform):
        self.samples = [(base_ds.samples[i][0], base_ds.samples[i][1]) for i in indices]
        self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

generator = torch.Generator().manual_seed(42)
idx = torch.randperm(len(base_dataset), generator=generator).tolist()
train_dataset = TransformSubset(base_dataset, idx[:train_size], train_transform)
val_dataset = TransformSubset(base_dataset, idx[train_size:train_size+val_size], test_transform)
test_dataset = TransformSubset(base_dataset, idx[train_size+val_size:], test_transform)
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

# ==================== Utility Functions ====================
class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.001):
        self.patience, self.min_delta = patience, min_delta
        self.counter, self.best_loss, self.best_model = 0, None, None
    def __call__(self, val_loss, model):
        if self.best_loss is None or val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.best_model = copy.deepcopy(model.state_dict())
            self.counter = 0
        else:
            self.counter += 1
        return self.counter >= self.patience

def train_one_epoch(model, criterion, optimizers, loader):
    model.train()
    rloss, correct, total = 0.0, 0, 0
    if not isinstance(optimizers, list): optimizers = [optimizers]
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        for o in optimizers: o.zero_grad()
        out = model(imgs); loss = criterion(out, labels)
        loss.backward()
        for o in optimizers: o.step()
        rloss += loss.item()*imgs.size(0)
        _, pred = torch.max(out, 1); total += labels.size(0); correct += (pred==labels).sum().item()
    return rloss/total, 100*correct/total

def evaluate_model(model, criterion, loader):
    model.eval()
    rloss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs); loss = criterion(out, labels)
            rloss += loss.item()*imgs.size(0)
            _, pred = torch.max(out, 1); total += labels.size(0); correct += (pred==labels).sum().item()
    return rloss/total, 100*correct/total

def get_predictions(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for imgs, lbl in loader:
            _, p = torch.max(model(imgs.to(device)), 1)
            preds.extend(p.cpu().numpy()); labels.extend(lbl.numpy())
    return np.array(labels), np.array(preds)

def plot_cm(cm, classes, title, ax):
    pct = cm.astype('float') / cm.sum(axis=1, keepdims=True) * 100
    ax.imshow(cm, cmap=plt.cm.Blues)
    ax.set_title(title, fontsize=11, fontweight='bold')
    t = np.arange(len(classes))
    ax.set_xticks(t); ax.set_xticklabels(classes, rotation=45, ha='right')
    ax.set_yticks(t); ax.set_yticklabels(classes)
    th = cm.max()/2
    for i,j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        ax.text(j,i,f'{cm[i,j]}\n({pct[i,j]:.0f}%)', ha='center', va='center',
                color='white' if cm[i,j]>th else 'black')
    ax.set_ylabel('True'); ax.set_xlabel('Predicted')

print("Data loaded. Utilities ready.")

## Section 1: Fix ResNet18 Overfitting

**Problem (from MH1):** ResNet18 had a 5.32% overfitting gap (97.59% train vs 92.26% val). Val accuracy peaked at epoch 8 (95.83%) then degraded to 92.26% by epoch 15.

**Solution — two techniques combined:**
1. **Data Augmentation** — RandomHorizontalFlip, RandomRotation, ColorJitter increase training variety, forcing the model to learn generalizable features instead of memorizing specific images.
2. **Early Stopping** (patience=3) — monitors validation loss and restores the best model when training starts hurting generalization. No more manually guessing the right epoch count.

We freeze all pretrained layers and only train the FC head (512 → 4), same as MH1.

In [0]:
# ==================== Build ResNet18 ====================
resnet_model = models.resnet18(pretrained=True)
for param in resnet_model.parameters():
    param.requires_grad = False
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, 4)
resnet_model = resnet_model.to(device)

resnet_criterion = nn.CrossEntropyLoss()
resnet_optimizer = optim.Adam(resnet_model.fc.parameters(), lr=0.001)
early_stop_resnet = EarlyStopping(patience=3)

# ==================== Train ====================
resnet_tl, resnet_vl, resnet_ta, resnet_va = [], [], [], []

print("=" * 60)
print("Training ResNet18 (Augmented + Early Stopping)")
print("=" * 60)

start = time.time()
for epoch in range(20):
    tl, ta = train_one_epoch(resnet_model, resnet_criterion, resnet_optimizer, train_loader)
    vl, va = evaluate_model(resnet_model, resnet_criterion, val_loader)
    resnet_tl.append(tl); resnet_vl.append(vl); resnet_ta.append(ta); resnet_va.append(va)
    print(f"Epoch [{epoch+1:2d}/20] | Train: {tl:.4f} / {ta:.2f}% | Val: {vl:.4f} / {va:.2f}%")
    if early_stop_resnet(vl, resnet_model):
        print(f"\nEarly stopping at epoch {epoch+1}! Best val loss: {early_stop_resnet.best_loss:.4f}")
        resnet_model.load_state_dict(early_stop_resnet.best_model)
        break
resnet_time = time.time() - start
print(f"Training time: {resnet_time:.1f}s")

# ==================== Plot ====================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, len(resnet_tl)+1)
best_ep = len(resnet_tl) - early_stop_resnet.counter
ax1.plot(ep, resnet_tl, 'b-o', label='Train'); ax1.plot(ep, resnet_vl, 'r-s', label='Val')
ax1.axvline(x=best_ep, color='green', ls='--', label=f'Best (ep {best_ep})')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(True)
ax2.plot(ep, resnet_ta, 'b-o', label='Train'); ax2.plot(ep, resnet_va, 'r-s', label='Val')
ax2.axvline(x=best_ep, color='green', ls='--', label=f'Best (ep {best_ep})')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(True)
plt.suptitle('Section 1: ResNet18 — Augmented + Early Stopping', fontweight='bold')
plt.tight_layout(); plt.show()

# ==================== Test ====================
_, resnet_test_acc = evaluate_model(resnet_model, resnet_criterion, test_loader)
resnet_gap = resnet_ta[-1] - resnet_va[best_ep-1]
print(f"\nAugmented ResNet18 Test Accuracy: {resnet_test_acc:.2f}%")
print(f"Overfit gap at best epoch: {resnet_gap:.2f}% (was 5.32% in MH1)")
print(f"Improvement over MH1: {resnet_test_acc - 95.29:+.2f}% (MH1 was 95.29%)")

## Section 2: Boost VGG16 Past 95%

**Problem (from MH1):** VGG16 had excellent generalization (1.58% gap) but test accuracy was 94.71%, just under the 95% hackathon target.

**Solution — Fine-tune the last convolutional block:**

Instead of only training the classifier head, we **unfreeze `features[24:]`** (the last conv block with 3 conv layers). This allows the model to adapt its high-level feature detectors specifically to weather patterns, while keeping early/mid layers (edges, textures) frozen.

**Why two optimizers?**
- `SGD(lr=0.0001, momentum=0.9)` for unfrozen conv layers — very small learning rate to avoid destroying pretrained features
- `Adam(lr=0.001)` for the new classifier head — higher rate since it trains from scratch

This **discriminative learning rate** strategy is standard in fine-tuning: deeper pretrained layers learn slowly, new layers learn fast.

In [0]:
# ==================== Build VGG16 with partial unfreezing ====================
vgg_model = models.vgg16(pretrained=True)
for param in vgg_model.parameters():
    param.requires_grad = False

# Unfreeze last conv block (features[24:] = 3 conv layers)
for param in vgg_model.features[24:].parameters():
    param.requires_grad = True

vgg_model.classifier[6] = nn.Linear(4096, 4)
vgg_model = vgg_model.to(device)

vgg_criterion = nn.CrossEntropyLoss()
opt_conv = optim.SGD(vgg_model.features[24:].parameters(), lr=0.0001, momentum=0.9)
opt_cls = optim.Adam(vgg_model.classifier[6].parameters(), lr=0.001)
early_stop_vgg = EarlyStopping(patience=3)

trainable = sum(p.numel() for p in vgg_model.parameters() if p.requires_grad)
print(f"VGG16 trainable parameters: {trainable:,} (was 16,388 in MH1 — now includes last conv block)")

# ==================== Train ====================
vgg_tl, vgg_vl, vgg_ta, vgg_va = [], [], [], []

print("\n" + "=" * 60)
print("Training VGG16 (Fine-tuned Last Conv Block)")
print("=" * 60)

start = time.time()
for epoch in range(20):
    tl, ta = train_one_epoch(vgg_model, vgg_criterion, [opt_conv, opt_cls], train_loader)
    vl, va = evaluate_model(vgg_model, vgg_criterion, val_loader)
    vgg_tl.append(tl); vgg_vl.append(vl); vgg_ta.append(ta); vgg_va.append(va)
    print(f"Epoch [{epoch+1:2d}/20] | Train: {tl:.4f} / {ta:.2f}% | Val: {vl:.4f} / {va:.2f}%")
    if early_stop_vgg(vl, vgg_model):
        print(f"\nEarly stopping at epoch {epoch+1}! Best val loss: {early_stop_vgg.best_loss:.4f}")
        vgg_model.load_state_dict(early_stop_vgg.best_model)
        break
vgg_time = time.time() - start
print(f"Training time: {vgg_time:.1f}s")

# ==================== Plot ====================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, len(vgg_tl)+1)
best_ep_v = len(vgg_tl) - early_stop_vgg.counter
ax1.plot(ep, vgg_tl, 'b-o', label='Train'); ax1.plot(ep, vgg_vl, 'r-s', label='Val')
ax1.axvline(x=best_ep_v, color='green', ls='--', label=f'Best (ep {best_ep_v})')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(True)
ax2.plot(ep, vgg_ta, 'b-o', label='Train'); ax2.plot(ep, vgg_va, 'r-s', label='Val')
ax2.axvline(x=best_ep_v, color='green', ls='--', label=f'Best (ep {best_ep_v})')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(True)
plt.suptitle('Section 2: VGG16 — Fine-tuned Last Conv Block', fontweight='bold')
plt.tight_layout(); plt.show()

# ==================== Test ====================
_, vgg_test_acc = evaluate_model(vgg_model, vgg_criterion, test_loader)
print(f"\nFine-tuned VGG16 Test Accuracy: {vgg_test_acc:.2f}%")
print(f"Improvement over MH1: {vgg_test_acc - 94.71:+.2f}% (MH1 was 94.71%)")

## Section 3: Ensemble — Combining Both Models

**Concept:** Instead of picking one model, we ask BOTH for their opinion and **average their softmax probabilities** for each class. The final prediction is whichever class has the highest averaged confidence.

**Why this works for our models (from MH1 per-class results):**

| Class | ResNet18 | VGG16 | Who helps? |
|---------|----------|-------|------------|
| Cloudy | 97.2% | 88.9% | ResNet18 rescues VGG16 |
| Rain | 97.0% | 100% | VGG16 rescues ResNet18 |
| Shine | 85.4% | 97.6% | VGG16 rescues ResNet18 |
| Sunrise | 100% | 93.3% | ResNet18 rescues VGG16 |

When one model is uncertain, the other gives a confident correct prediction. Averaging lets each model compensate for the other's weakness. **No retraining needed.**

In [0]:
# ==================== Ensemble Prediction Function ====================
def ensemble_predict(models_list, loader):
    """Average softmax probabilities from multiple models."""
    for m in models_list: m.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            avg = torch.zeros(imgs.size(0), 4).to(device)
            for m in models_list:
                avg += F.softmax(m(imgs), dim=1)
            avg /= len(models_list)
            all_probs.append(avg.cpu()); all_labels.append(labels)
    all_probs = torch.cat(all_probs); all_labels = torch.cat(all_labels)
    _, preds = torch.max(all_probs, 1)
    return all_labels.numpy(), preds.numpy(), all_probs

# ==================== Get All Predictions ====================
resnet_true, resnet_pred = get_predictions(resnet_model, test_loader)
vgg_true, vgg_pred = get_predictions(vgg_model, test_loader)
ens_true, ens_pred, ens_probs = ensemble_predict([resnet_model, vgg_model], test_loader)

resnet_acc = 100 * (resnet_true == resnet_pred).sum() / len(resnet_true)
vgg_acc = 100 * (vgg_true == vgg_pred).sum() / len(vgg_true)
ensemble_acc = 100 * (ens_true == ens_pred).sum() / len(ens_true)

print(f"ResNet18 Test:  {resnet_acc:.2f}%")
print(f"VGG16 Test:     {vgg_acc:.2f}%")
print(f"Ensemble Test:  {ensemble_acc:.2f}%")
print(f"Ensemble gain:  {ensemble_acc - max(resnet_acc, vgg_acc):+.2f}% over best single model")

# ==================== Confusion Matrices ====================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_cm(confusion_matrix(resnet_true, resnet_pred), class_names, 'ResNet18', axes[0])
plot_cm(confusion_matrix(vgg_true, vgg_pred), class_names, 'VGG16', axes[1])
plot_cm(confusion_matrix(ens_true, ens_pred), class_names, 'Ensemble', axes[2])
plt.suptitle('Test Set — Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# ==================== Per-Class Comparison ====================
print(f"\n{'='*65}")
print(f"{'PER-CLASS ACCURACY COMPARISON':^65}")
print(f"{'='*65}")
print(f"{'Class':<12} {'ResNet18':<12} {'VGG16':<12} {'Ensemble':<12} {'Winner':<12}")
print(f"{'-'*65}")
for i, cls in enumerate(class_names):
    mask = resnet_true == i
    r = 100*(resnet_pred[mask]==i).sum()/mask.sum()
    v = 100*(vgg_pred[mask]==i).sum()/mask.sum()
    e = 100*(ens_pred[mask]==i).sum()/mask.sum()
    best = max(r,v,e)
    w = 'Ensemble' if e==best else ('ResNet18' if r==best else 'VGG16')
    print(f"{cls:<12} {r:<12.1f} {v:<12.1f} {e:<12.1f} {w:<12}")
print(f"{'='*65}")

## Misclassified Images: What Fools Each Model?

This is one of the most insightful debugging tools — it shows you *exactly* what confuses each model.

We visualize three categories:
1. **Ensemble CORRECTED** — images that ResNet18 or VGG16 got wrong, but the ensemble fixed by averaging their confidences. This proves why ensembles work.
2. **Ensemble STILL WRONG** — images even the ensemble can't classify. These are genuinely ambiguous (e.g., a cloudy sunset that looks like sunrise). Understanding these helps you decide if more data or better augmentation is needed.

Each image shows:
- **True label** in the title
- **R** = ResNet18 prediction, **V** = VGG16 prediction, **E** = Ensemble prediction
- Green text = correct, Red text = wrong

In [0]:
# ==================== Inverse normalization for display ====================
inv_norm = transforms.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)

# ==================== Classify misclassification types ====================
resnet_wrong = resnet_pred != resnet_true
vgg_wrong = vgg_pred != vgg_true
ens_wrong = ens_pred != ens_true

# Category 1: Ensemble FIXED (at least one model wrong, ensemble correct)
fixed_mask = (resnet_wrong | vgg_wrong) & ~ens_wrong
fixed_idx = np.where(fixed_mask)[0]

# Category 2: Ensemble STILL WRONG
still_wrong_idx = np.where(ens_wrong)[0]

print(f"Total test images: {len(ens_true)}")
print(f"ResNet18 errors:       {resnet_wrong.sum()}")
print(f"VGG16 errors:          {vgg_wrong.sum()}")
print(f"Ensemble errors:       {ens_wrong.sum()}")
print(f"Ensemble CORRECTED:    {len(fixed_idx)} images (was wrong, now right)")
print(f"Ensemble STILL WRONG:  {len(still_wrong_idx)} images")

# ==================== Visualization helper ====================
def show_misclassified(indices, suptitle, max_show=10):
    """Display misclassified images with per-model predictions."""
    n = min(len(indices), max_show)
    if n == 0:
        print(f"\n{suptitle}: No images found!")
        return
    cols = min(n, 5)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.8, rows * 4.5))
    if n == 1:
        axes = np.array([[axes]])
    elif rows == 1:
        axes = axes.reshape(1, -1)

    for i in range(n):
        idx = indices[i]
        img, _ = test_dataset[idx]
        display_img = inv_norm(img).permute(1, 2, 0).numpy().clip(0, 1)
        r, c = divmod(i, cols)
        ax = axes[r, c]
        ax.imshow(display_img)

        true_lbl = class_names[ens_true[idx]]
        r_lbl = class_names[resnet_pred[idx]]
        v_lbl = class_names[vgg_pred[idx]]
        e_lbl = class_names[ens_pred[idx]]

        ax.set_title(f"True: {true_lbl}", fontsize=11, fontweight='bold')

        # Color code: green=correct, red=wrong
        r_color = 'green' if resnet_pred[idx] == ens_true[idx] else 'red'
        v_color = 'green' if vgg_pred[idx] == ens_true[idx] else 'red'
        e_color = 'green' if ens_pred[idx] == ens_true[idx] else 'red'

        # Build annotation text with colors
        ax.text(0.02, -0.08, f"R: {r_lbl}", transform=ax.transAxes,
                fontsize=9, color=r_color, fontweight='bold')
        ax.text(0.35, -0.08, f"V: {v_lbl}", transform=ax.transAxes,
                fontsize=9, color=v_color, fontweight='bold')
        ax.text(0.68, -0.08, f"E: {e_lbl}", transform=ax.transAxes,
                fontsize=9, color=e_color, fontweight='bold')
        ax.axis('off')

    # Hide empty axes
    for i in range(n, rows * cols):
        r, c = divmod(i, cols)
        axes[r, c].axis('off')

    fig.suptitle(f"{suptitle} ({len(indices)} total, showing {n})",
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# ==================== 1. Ensemble CORRECTED ====================
print("\n" + "=" * 60)
print("IMAGES THE ENSEMBLE CORRECTED")
print("(At least one model was wrong, but ensemble got it right)")
print("=" * 60)
show_misclassified(fixed_idx, "Ensemble CORRECTED")

# Show which model was wrong for each corrected image
if len(fixed_idx) > 0:
    print(f"\nCorrection breakdown:")
    only_resnet_wrong = ((resnet_wrong) & ~vgg_wrong & ~ens_wrong)
    only_vgg_wrong = ((vgg_wrong) & ~resnet_wrong & ~ens_wrong)
    both_wrong_fixed = (resnet_wrong & vgg_wrong & ~ens_wrong)
    print(f"  Only ResNet18 wrong (VGG16 saved it):  {only_resnet_wrong.sum()}")
    print(f"  Only VGG16 wrong (ResNet18 saved it):   {only_vgg_wrong.sum()}")
    print(f"  BOTH wrong but ensemble still fixed:    {both_wrong_fixed.sum()}")

# ==================== 2. Ensemble STILL WRONG ====================
print("\n" + "=" * 60)
print("IMAGES THE ENSEMBLE STILL GETS WRONG")
print("(Even averaging both models can't save these)")
print("=" * 60)
show_misclassified(still_wrong_idx, "Ensemble STILL WRONG")

# Analyze failure patterns
if len(still_wrong_idx) > 0:
    print(f"\nFailure pattern analysis:")
    print(f"{'True Class':<12} {'Predicted As':<12} {'Count':<8}")
    print(f"{'-'*35}")
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            if i == j: continue
            mask = (ens_true == i) & (ens_pred == j)
            if mask.sum() > 0:
                print(f"{class_names[i]:<12} {class_names[j]:<12} {mask.sum():<8}")
    
    # Check if both models also failed on these
    both_also_wrong = resnet_wrong[still_wrong_idx] & vgg_wrong[still_wrong_idx]
    print(f"\nOf {len(still_wrong_idx)} ensemble failures:")
    print(f"  Both models also wrong: {both_also_wrong.sum()} (genuinely ambiguous)")
    print(f"  At least one model right: {len(still_wrong_idx) - both_also_wrong.sum()} (ensemble averaging hurt)")

## Section 4: Knowledge Distillation

**The problem with ensembles:** Running two models doubles compute and memory at inference.

**Solution:** Train a small **student model** to mimic the ensemble's predictions. The student learns from the ensemble's "soft labels" (probability distributions), which carry more information than hard labels (0 or 1).

**How it works:**
1. The ensemble (teacher) produces soft predictions with **temperature scaling** (T=3). High temperature makes probabilities smoother, revealing inter-class relationships (e.g., "60% rain, 30% cloudy" teaches the student about rain-cloud visual similarity).
2. The student minimizes **KL Divergence** between its softened output and the teacher's soft output.
3. We also add standard CrossEntropy loss with true labels for stability (weighted by α=0.7 for distillation, 0.3 for hard labels).

**Student architecture — a lightweight 2-layer CNN:**
```
Conv(3→32, 3×3) → ReLU → MaxPool(2)
Conv(32→64, 3×3) → ReLU → MaxPool(2)
FC(64×54×54 → 128) → FC(128 → 4)
```

> **Note:** The FC layer (64×54×54 = 186K inputs) dominates parameter count (~24M). Adding an `AdaptiveAvgPool` before FC would reduce this dramatically — a good exercise to try!

In [0]:
# ==================== Student Model ====================
class StudentCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 54 * 54, 128), nn.ReLU(), nn.Linear(128, num_classes),
        )
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

student = StudentCNN(num_classes=4).to(device)
student_params = sum(p.numel() for p in student.parameters())
print(f"Student parameters: {student_params:,}")
print(f"vs Ensemble (ResNet18+VGG16): ~145.5M parameters")
print(f"Compression ratio: {145.5e6/student_params:.1f}x smaller")

# ==================== Distillation Training ====================
temperature = 3.0
alpha = 0.7  # 70% distillation loss, 30% hard label loss
student_opt = optim.Adam(student.parameters(), lr=0.001)
criterion_hard = nn.CrossEntropyLoss()
early_stop_student = EarlyStopping(patience=3)

resnet_model.eval(); vgg_model.eval()
student_tl, student_vl, student_ta, student_va = [], [], [], []

print("\n" + "=" * 60)
print(f"Training Student via Knowledge Distillation (T={temperature}, alpha={alpha})")
print("=" * 60)

start = time.time()
for epoch in range(20):
    student.train()
    rloss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        # Teacher soft predictions
        with torch.no_grad():
            t_logits = (resnet_model(imgs) + vgg_model(imgs)) / 2
            t_soft = F.softmax(t_logits / temperature, dim=1)
        # Student forward
        s_logits = student(imgs)
        s_soft = F.log_softmax(s_logits / temperature, dim=1)
        # Combined loss
        loss_kd = F.kl_div(s_soft, t_soft, reduction='batchmean') * (temperature**2)
        loss_hard = criterion_hard(s_logits, labels)
        loss = alpha * loss_kd + (1 - alpha) * loss_hard
        student_opt.zero_grad(); loss.backward(); student_opt.step()
        rloss += loss.item()*imgs.size(0)
        _, pred = torch.max(s_logits, 1); total += labels.size(0); correct += (pred==labels).sum().item()
    
    tl = rloss/total; ta = 100*correct/total
    vl, va = evaluate_model(student, criterion_hard, val_loader)
    student_tl.append(tl); student_vl.append(vl); student_ta.append(ta); student_va.append(va)
    print(f"Epoch [{epoch+1:2d}/20] | Train: {tl:.4f} / {ta:.2f}% | Val: {vl:.4f} / {va:.2f}%")
    if early_stop_student(vl, student):
        print(f"\nEarly stopping at epoch {epoch+1}!")
        student.load_state_dict(early_stop_student.best_model)
        break
student_time = time.time() - start

# ==================== Evaluate Student ====================
_, student_test_acc = evaluate_model(student, criterion_hard, test_loader)

print(f"\n{'='*60}")
print(f"{'DISTILLATION RESULTS':^60}")
print(f"{'='*60}")
print(f"{'Model':<22} {'Test Acc':<12} {'Parameters':<15}")
print(f"{'-'*60}")
print(f"{'Ensemble (teacher)':<22} {ensemble_acc:<12.2f} {'~145.5M':<15}")
print(f"{'Student (distilled)':<22} {student_test_acc:<12.2f} {student_params:,}")
print(f"{'='*60}")
print(f"Knowledge transfer: {student_test_acc/ensemble_acc*100:.1f}% of teacher accuracy")
print(f"Size reduction: {145.5e6/student_params:.1f}x smaller")

## Section 5: VGG16 Pruned at 70% with Fine-tuning

**From MH1 pruning results:** VGG16 at 70% pruning maintained 95.29% accuracy (even improved!) — but that was **without** fine-tuning after pruning.

**Here we add post-pruning fine-tuning** — 5 epochs of training after pruning to let the remaining weights readjust. This typically recovers 1-3% of any accuracy lost.

**Process:**
1. Deep-copy the trained VGG16 model
2. Apply **70% global L1 unstructured pruning** (remove smallest-magnitude weights across all layers)
3. Make pruning permanent (remove masks)
4. **Fine-tune for 5 epochs** with a small learning rate (0.0001)
5. Compare: before pruning → after pruning → after fine-tuning

In [0]:
# ==================== Prune VGG16 ====================
vgg_pruned = copy.deepcopy(vgg_model)
_, vgg_before_prune = evaluate_model(vgg_pruned, vgg_criterion, test_loader)

# Apply 70% global L1 pruning
params_to_prune = []
for name, module in vgg_pruned.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        params_to_prune.append((module, 'weight'))
prune.global_unstructured(params_to_prune, pruning_method=prune.L1Unstructured, amount=0.7)

_, vgg_after_prune = evaluate_model(vgg_pruned, vgg_criterion, test_loader)

# Make pruning permanent
for name, module in vgg_pruned.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        try: prune.remove(module, 'weight')
        except ValueError: pass

total_p = sum(p.numel() for p in vgg_pruned.parameters())
nonzero_p = sum((p!=0).sum().item() for p in vgg_pruned.parameters())
sparsity = 100 * (1 - nonzero_p/total_p)
print(f"Sparsity: {sparsity:.1f}% ({nonzero_p:,} / {total_p:,} non-zero)")
print(f"Before pruning: {vgg_before_prune:.2f}%")
print(f"After pruning (no fine-tune): {vgg_after_prune:.2f}%")

# ==================== Fine-tune after pruning ====================
for param in vgg_pruned.parameters():
    param.requires_grad = True
prune_opt = optim.Adam(vgg_pruned.parameters(), lr=0.0001)
prune_crit = nn.CrossEntropyLoss()

print(f"\n{'='*60}")
print("Fine-tuning pruned VGG16 (5 epochs)")
print(f"{'='*60}")
for epoch in range(5):
    tl, ta = train_one_epoch(vgg_pruned, prune_crit, prune_opt, train_loader)
    vl, va = evaluate_model(vgg_pruned, prune_crit, val_loader)
    print(f"Epoch [{epoch+1}/5] | Train: {tl:.4f} / {ta:.2f}% | Val: {vl:.4f} / {va:.2f}%")

_, vgg_pruned_acc = evaluate_model(vgg_pruned, prune_crit, test_loader)

print(f"\n{'='*60}")
print(f"{'VGG16 PRUNING RESULTS':^60}")
print(f"{'='*60}")
print(f"{'Stage':<35} {'Test Acc':<15}")
print(f"{'-'*60}")
print(f"{'Before pruning':<35} {vgg_before_prune:.2f}%")
print(f"{'After 70% pruning (no fine-tune)':<35} {vgg_after_prune:.2f}%")
print(f"{'After pruning + 5-epoch fine-tune':<35} {vgg_pruned_acc:.2f}%")
print(f"{'='*60}")
print(f"Recovery from fine-tuning: {vgg_pruned_acc - vgg_after_prune:+.2f}%")
print(f"Active parameters: {nonzero_p:,} ({100-sparsity:.1f}% of original {total_p:,})")

## Final Comparison: All Approaches

Below we compare **every model** from both notebooks (MH1 baselines + MH2 advanced) on:
- **Test Accuracy** — classification performance on unseen weather images
- **Parameters** — model complexity and memory footprint
- **Inference Speed** — time to process the full test set on GPU
- **Recommendation** — which model to use for different deployment scenarios

In [0]:
# ==================== Measure Inference Speed ====================
def measure_speed(model, loader, runs=3):
    model.eval()
    times = []
    with torch.no_grad():
        for _ in range(runs):
            t0 = time.time()
            for imgs, _ in loader: _ = model(imgs.to(device))
            times.append(time.time()-t0)
    return np.mean(times)

def measure_ensemble_speed(models, loader, runs=3):
    for m in models: m.eval()
    times = []
    with torch.no_grad():
        for _ in range(runs):
            t0 = time.time()
            for imgs, _ in loader:
                imgs = imgs.to(device)
                for m in models: _ = m(imgs)
            times.append(time.time()-t0)
    return np.mean(times)

def get_size(model):
    with tempfile.NamedTemporaryFile(delete=True, suffix='.pt') as f:
        torch.save(model.state_dict(), f.name)
        return os.path.getsize(f.name)/(1024*1024)

resnet_speed = measure_speed(resnet_model, test_loader)
vgg_speed = measure_speed(vgg_model, test_loader)
student_speed = measure_speed(student, test_loader)
pruned_speed = measure_speed(vgg_pruned, test_loader)
ensemble_speed = measure_ensemble_speed([resnet_model, vgg_model], test_loader)

# ==================== Final Table ====================
models_info = [
    ("Original ResNet18*",   95.29,            "11.2M",    "-"),
    ("Original VGG16*",      94.71,            "134.3M",   "-"),
    ("Aug+ES ResNet18",      resnet_test_acc,  f"{sum(p.numel() for p in resnet_model.parameters())/1e6:.1f}M", f"{resnet_speed:.3f}s"),
    ("Fine-tuned VGG16",     vgg_test_acc,     f"{sum(p.numel() for p in vgg_model.parameters())/1e6:.1f}M",  f"{vgg_speed:.3f}s"),
    ("Ensemble",             ensemble_acc,     "~145.5M",  f"{ensemble_speed:.3f}s"),
    ("Student (Distilled)",  student_test_acc, f"{student_params/1e6:.1f}M", f"{student_speed:.3f}s"),
    ("VGG16 Pruned 70%",     vgg_pruned_acc,   f"{nonzero_p/1e6:.1f}M active", f"{pruned_speed:.3f}s"),
]

print(f"\n{'='*80}")
print(f"{'FINAL MODEL COMPARISON':^80}")
print(f"{'='*80}")
print(f"{'Model':<22} {'Test Acc':<11} {'Parameters':<17} {'Inference':<12}")
print(f"{'-'*80}")
for name, acc, params, speed in models_info:
    print(f"{name:<22} {acc:<11.2f} {params:<17} {speed:<12}")
print(f"{'='*80}")
print("* MH1 baselines (no augmentation, no fine-tuning, no early stopping)")

# ==================== Bar Chart ====================
fig, ax = plt.subplots(figsize=(12, 6))
names = [m[0] for m in models_info]
accs = [m[1] for m in models_info]
colors = ['#95a5a6','#95a5a6','#2ca02c','#d62728','#9467bd','#8c564b','#e377c2']
bars = ax.bar(names, accs, color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(y=95, color='red', ls='--', alpha=0.7, label='95% target')
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Final Model Comparison — All Approaches', fontsize=14, fontweight='bold')
ax.set_ylim(min(accs)-5, 102)
plt.xticks(rotation=30, ha='right')
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=9)
ax.legend(fontsize=11); plt.tight_layout(); plt.show()

# ==================== Recommendations ====================
print(f"""
{'='*65}
RECOMMENDATIONS
{'='*65}
1. BEST ACCURACY:    Ensemble ({ensemble_acc:.2f}%)
   Use when accuracy is critical and compute budget allows.

2. BEST BALANCE:     Fine-tuned VGG16 ({vgg_test_acc:.2f}%)
   Single model, strong generalization, fine-tuned conv layers.

3. FASTEST DEPLOY:   Student CNN ({student_test_acc:.2f}%)
   {student_params/1e6:.1f}M params, fastest inference, edge/mobile ready.

4. COST EFFECTIVE:   Aug+ES ResNet18 ({resnet_test_acc:.2f}%)
   Lightweight with augmentation fixing the overfitting issue.

5. COMPRESSION:      Pruned VGG16 ({vgg_pruned_acc:.2f}%)
   70% weights removed + fine-tuned for accuracy recovery.
{'='*65}
""")

## Observations & Key Learnings

### Results Summary (Actual Colab T4 GPU Results)

| Model | Test Acc | Val Acc | Overfit Gap | Notes |
|-------|----------|---------|-------------|-------|
| MH1 ResNet18 (baseline) | 95.29% | 92.26% | 5.32% | FC-only, no augmentation |
| MH1 VGG16 (baseline) | 94.71% | 97.02% | 1.58% | FC-only, no augmentation |
| MH2 Aug+ES ResNet18 | 92.35% | 94.05% | 2.90% | Augmentation reduced gap but hurt accuracy |
| MH2 Fine-tuned VGG16 | 94.71% | 97.62% | ~0.7% | Best generalization, unfrozen last conv block |
| MH2 Ensemble | 95.29% | — | — | Averaged softmax, complementary strengths |
| MH2 Student (Distilled) | 91.18% | 95.24% | — | 2-layer CNN, 23.9M params, KL divergence |
| **MH2 VGG16 Pruned 70%** | **95.88%** | — | — | **Best overall — pruning as regularization** |

### Key Learnings

**1. Data augmentation is NOT always beneficial**
- With frozen feature extractors (FC-only training), augmentation *hurts* because ImageNet features can't adapt to rotated/color-shifted weather images.
- The overfitting gap improved (5.32% → 2.90%) but absolute accuracy dropped (95.29% → 92.35%).
- **Lesson:** Augmentation needs unfrozen layers to be effective.

**2. Fine-tuning > FC-only training**
- Unfreezing VGG16's last conv block (`features[24:]`) with discriminative learning rates (SGD 0.0001 for conv, Adam 0.001 for classifier) gave the best single-model generalization (97.62% val, 0.7% gap).
- **Lesson:** Pretrained features are a starting point, not the final answer — domain adaptation of deeper layers matters.

**3. Ensemble works through complementary weaknesses**
- ResNet18 strong on Cloudy (93.0%) and Sunrise (100%); VGG16 strong on Rain (96.7%) and Shine (91.9%).
- Ensemble achieved 95.29% by letting each model rescue the other.
- VGG16 saved ResNet18 in **9 out of 10** corrected images (mostly Shine class).
- **Lesson:** Diversity in model architecture matters more than individual accuracy for ensembles.

**4. Knowledge distillation has limits with simple architectures**
- Student CNN (23.9M params) only reached 91.18% — worse than both teachers.
- The FC layer (64×54×54 → 128) dominated parameter count while lacking the inductive biases of pretrained convolutions.
- Val accuracy (95.24%) was strong but didn't transfer to test — the student memorized soft-label patterns.
- **Lesson:** Distillation needs a student with sufficient architectural capacity. A pretrained MobileNetV2 would be a better student choice.

**5. Pruning is the most powerful optimization (counter-intuitive finding)**
- VGG16 at 70% sparsity + 5-epoch fine-tuning achieved **95.88%** — the highest accuracy across ALL approaches.
- Removing 94M of 134M parameters acted as **regularization**, eliminating redundant weights that slightly overfitted.
- This single pruned model outperformed the two-model ensemble (95.88% vs 95.29%).
- **Lesson:** Bigger models aren't always better. Pruning removes noise, not signal, especially in transfer learning where most pretrained features are irrelevant to the target domain.

**6. Misclassified image analysis reveals dataset limitations**
- 8 images remained misclassified even by the ensemble.
- Most failures were Cloudy ↔ Shine confusion (overcast bright skies) and Cloudy ↔ Sunrise (warm-toned clouds).
- These represent **labeling ambiguity** — the images are genuinely at the boundary between classes.
- **Lesson:** Model accuracy has a ceiling determined by data quality. Beyond ~96% on this dataset, improvements require better class definitions or additional context (time of day, metadata).

### Recommended Production Model

**VGG16 Pruned at 70%** (95.88% accuracy, 40.3M active parameters)
- Best accuracy across all approaches
- Single model — no ensemble overhead
- Can be further optimized with ONNX export + INT8 quantization for edge deployment
- Fine-tuning after pruning recovered any accuracy loss and then some

### Next Steps
- ONNX export + quantization for real file size compression
- Real-world validation: model prediction vs weather API ground truth
- Streamlit/Gradio demo app for live weather classification
- Try MobileNetV2 as a better student for knowledge distillation

In [0]:
# ======================================================================
# Save and download the best model (VGG16 Pruned 70% — 95.88% accuracy)
# Run this cell on Colab to download the .pt file to your local machine
# ======================================================================
from google.colab import files

# Save model state dict
model_path = 'vgg16_pruned_70_weather_classifier.pt'
torch.save({
    'model_state_dict': vgg_pruned.state_dict(),
    'class_names': class_names,
    'accuracy': vgg_pruned_acc,
    'sparsity': sparsity,
    'active_params': nonzero_p,
    'total_params': total_p,
    'architecture': 'VGG16 with 70% global L1 pruning + 5-epoch fine-tuning',
    'input_size': (3, 224, 224),
    'normalize_mean': [0.485, 0.456, 0.406],
    'normalize_std': [0.229, 0.224, 0.225],
}, model_path)

file_size = os.path.getsize(model_path) / (1024 * 1024)
print(f"Model saved: {model_path}")
print(f"File size: {file_size:.1f} MB")
print(f"Accuracy: {vgg_pruned_acc:.2f}%")
print(f"Sparsity: {sparsity:.1f}% ({nonzero_p:,} active / {total_p:,} total params)")
print(f"Classes: {class_names}")
print(f"\nDownloading to your local machine...")

# Trigger browser download
files.download(model_path)